## Challenges of AI-based models in spatial applications

**The motto of today's session is "Know the limits of your AI model".**

In particular, when working with geographic data, it's important to inspect your predictions from a *spatial* perspective. This applies to many AI applications, but is especially important when your data contains spatial dependencies between features which leads to spatial autocorrelation.

> AI algorithms have become very popular for spatial mapping of the environment due to their ability to fit nonlinear and complex relationships. However, this ability comes with the disadvantage that they can only be applied to new data if these are similar to the training data.


Whenever you have the presumption that observation that are located close to each other are more similar to each other than to more distant observations, then you need to start thinking about spatial autocorrelation and its effects.
Mainly, because this spatial dependency violates standard statistical techniques that assume independence among observations.


> Using naive random n-fold or leave-one-out cross-validation methods (or a simple random train-test split) to assess global model quality (usually equated with map accuracy) makes sense when the data are independent and identically distributed. When this is not the case, dependencies between nearby samples, e.g., in a spatial cluster, are ignored and result in biased, overly optimistic model assessment. Alternative cross-validation approaches such as spatial cross-validation that control for such dependencies are the only way to overcome this bias.

**To put in simple words: When you neglect the spatial structure in the data, then the model performance is very likely much lower than your performance metrics indicate (e.g., RMSE or R2 score).**


#### Further literature

* *Meyer, Hanna, and Edzer Pebesma. 2022. “Machine Learning-Based Global Maps of Ecological Variables and the Challenge of Assessing Them.” Nature Communications 13 (1): 2208. https://doi.org/10.1038/s41467-022-29838-9.*
* *Meyer, Hanna, and Edzer Pebesma. 2021. “Predicting into Unknown Space? Estimating the Area of Applicability of Spatial Prediction Models.” Methods in Ecology and Evolution 12 (9): 1620–33. https://doi.org/10.1111/2041-210X.13650.*
* *Nachtigall, Florian; Milojevic-Dupont, Nikola; Wagner, Felix; Creutzig, Felix. 2023. "Predicting building age from urban form at large scale" Computers, Environment and Urban Systems 105 (2023). https://doi.org/10.1016/j.compenvurbsys.2023.102010*



In [ ]:
from pathlib import Path

import category_encoders as ce
import folium
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
import xgboost as xgb
from shapely.geometry import Point
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

#ignore warnings from pandas
import warnings
warnings.filterwarnings("ignore")

## Download the data
Download the zip file `09_data.zip` from this [folder](https://tubcloud.tu-berlin.de/s/ZX6LbyAQzC5i6RL).
Unzip the folder and place it in a folder called `./data` in the same directory of this exercise.


## Inspect the data

For this exercise we will work with a dataset that is less related to Human-Water Systems, however, it is highly suitable to exemplify the impact of spatial dependencies. 
The dataset is of Airbnb listings and contains various features related to these listings, such as location, amenities, host information and Reviews. To avoid to spend too much time with data preprocessing (e.g., imputing missing features, encoding and conversion of temporal features into numeric values ), we provide a preprocessed version (`airbnb_dataset_preprocessed`) of the original dataset that can be found on [Kaggle](https://www.kaggle.com/datasets/rupindersinghrana/airbnb-price-dataset/data). Note, that no information about the variable units were passed with this dataset. 

### Task:
* Load the csv file into a pandas dataframe.
* Then transform the dataframe into a geodataframe by converting the columns `latitude` and `longitude` into a point geometry and by setting the coordinate reference system to EPSG:4326

# Data Preparation
We define a function to help us with data preparation and feature encoding.

In [ ]:
def data_preparation(df_train, df_test):
    # 1. Define your categorical columns
    categorical_cols = [
        'property_type', 'room_type', 'bed_type', 'cancellation_policy', 'city',
        'cleaning_fee', 'host_has_profile_pic', 'host_identity_verified',
        'instant_bookable'
    ]

    # 2. Initialize a TargetEncoder (with smoothing to stabilize rare categories)
    target_encoder = ce.TargetEncoder(
        cols=categorical_cols,
        smoothing=0.3,       # higher→stronger shrinkage toward global mean
        min_samples_leaf=20  # min samples to take category average seriously
    )

    # 3. Fit on TRAIN and transform both TRAIN and TEST
    df_train[[f"{c}_encoded" for c in categorical_cols]] = target_encoder.fit_transform(
        df_train[categorical_cols],
        df_train['log_price']
    )
    df_test[[f"{c}_encoded" for c in categorical_cols]] = target_encoder.transform(
        df_test[categorical_cols]
    )

    return df_train, df_test

# Random train test split
Let's start with the naive non-geographic approach.

### Task: 
* Run 6 iterations (> to make it comparable with the other model performances from attempt 1 and 2 <) and use a random split of train (80%) and test samples (20%).
* Use the list of features provided in the code cell below. Note that these also contain longitude and latitude.
* Let's fit a XGBoost model and predict the `log_price`, ie, the price of the AirBnB listings in its logarithmic form.
* Then assess performance using R² and Mean Squared Error (MSE).
* Finally, we calculate the average R² and MSE.

Note: We use in the exercise the R² as scoring measure, instead of other regression metrics, to better exemplify what happens when spatial dependencies in the data are disregarded.

In [ ]:
features = [
    'room_type_encoded',
    'bedrooms',
    'longitude',
    'latitude',
    'number_of_reviews',
    'accommodates',
    'bathrooms',
    'first_review_ts',
    'review_scores_rating',
    'property_type_encoded',
    'last_review_ts',
    'instant_bookable_encoded',
    'smoke detector'
]

In [ ]:
scores = []
results = {}

> *Wow! It looks like we got a pretty decent model with a R² score of about 0.70.
That's not bad, but should we trust these results?*

---

# Spatial train test split - Attempt 1

This time we will make it harder for our model.
Let's consider the spatial aspect and split our train and test samples according to the city the observations are located in.
In fact, we want to make sure that the test data should contain the samples within a single city. Training data should come from all other cities. Samples from the same city should only be present in one of the splits, not in both at the same time.

In easy words: We want to train our model with data from NYC, SF, DC, LA and Chicago. Then we want to test the model performance for Boston. We'll repeat this for all potential combinations. This will give us a first impression how good our model can transfer to other geographic areas.

* We run 6 iterations (> because we have 6 cities <) and use a spatial split of train and test samples (20%).
* Let's fit the XGBoost model and predict the `log_price`.
* Then we assess performance using R² and Mean Squared Error (MSE).
* Finally, we calculate the average R² and MSE.


You can use the code above and slightly adjust it.


In [ ]:

### > uncomment the code in this cell < 

# def train_test_split_spatial_city(df, city):
#     df_train = df[~(df["city"] == city)]
#     df_test = df[df["city"] == city]
#     return df_train, df_test

# cities = df.city.unique()
# print(cities)



In [ ]:
### > uncomment the code in this cell < 

# scores_spatial = []
# results_spatial = {}


# for i, city in enumerate(cities):

#     # city wise train test split
#     df_train, df_test = train_test_split_spatial_city(df, city)


#     df_train, df_test = data_preparation(df_train, df_test)


###   > add the rest of the code here < 

> What happened here. Our model performs considerably worse than before.

> In some cases we even got a negative R² score. This means that our model got it systematically wrong.

Let's try to fix that.

---

# Spatial train test split - Attempt 2

We overlooked one thing: The list of features contains `longitude` and `latitude`.
Nevertheless, it is very hard for our model to make sense of these coordinates when it has never seen similar coordinates before. Our model might capture the variations in LA, but has never seen any coordinates for Boston.

We can adjust the code above by simply removing these two attributes from our list of features.


In [ ]:

## < Uncomment the code in this cell >

# features = [
#     'room_type_encoded',
#     'bedrooms',
#     # 'longitude',  ######## removed #########
#     # 'latitude',   ######## removed #########
#     'number_of_reviews',
#     'accommodates',
#     'bathrooms',
#     'first_review_ts',
#     'review_scores_rating',
#     'property_type_encoded',
#     'last_review_ts',
#     'instant_bookable_encoded',
#     'smoke detector'
# ]
# results_spatial2 = {}

In [ ]:
## < Uncomment the code in this cell >


# scores_spatial2 = []
# results_spatial2 = {}

# for i, city in enumerate(cities):

#     # city wise train test split
#     df_train, df_test = train_test_split_spatial_city(df, city)


#     df_train, df_test = data_preparation(df_train, df_test)

###   > add the rest of the code here < 

Better... We couldn't reach the same performance as in our very first (non-spatial) approach, but at least we didn't get it completely wrong.

**Lesson learned #1:** Including coordinates or geographic information into your model silently assumes that your test and train samples cover the same area. In some situations this can be harmful.

**Lesson learned #2:** When you have a limited availability of training sample locations and strong spatial dependency, then be cautious about your model performance scores.

---


# Inspect the residuals

We will focus on a single city here for simplicity.
In theory and practice, you should inspect the residuals for all cities.



In [ ]:
city = "DC" # for DC the impacts of the different train test splits are best shown 

fig, axs = plt.subplots(1, 3, figsize=(15,5))


df_random_split = results[0]
df_city = df_random_split[df_random_split["city"] == city]

ax1 = axs[0]
ax1.scatter(
    df_city["prediction"],
    df_city["residual"],
    alpha=0.05,
)
ax1.axhline(
    0,
    color='red',
    linestyle='--'
)
ax1.set_title(f"{city}\nrandom split")
ax1.set_xlim((2, 8))
ax1.set_ylim((-3, 3))
ax1.set_xlabel('Predicted Values')
ax1.set_ylabel('Residuals (y_pred - y_test)')


ax2 = axs[1]
ax2.scatter(
    results_spatial[city]["prediction"],
    results_spatial[city]["residual"],
    alpha=0.05,
)
ax2.axhline(
    0,
    color='red',
    linestyle='--'
)
ax2.set_title(f"{city}\nspatial split - attempt 1")
ax2.set_xlim((2, 8))
ax2.set_ylim((-3, 3))
ax2.set_xlabel('Predicted Values')
ax2.set_ylabel('Residuals (y_pred - y_test)')


ax3 = axs[2]
ax3.scatter(
    results_spatial2[city]["prediction"],
    results_spatial2[city]["residual"],
    alpha=0.05,
)
ax3.axhline(
    0,
    color='red',
    linestyle='--'
)
ax3.set_title(f"{city}\nspatial split - attempt 2")
ax3.set_xlim((2, 8))
ax3.set_ylim((-3, 3))
ax3.set_xlabel('Predicted Values')
ax3.set_ylabel('Residuals (y_pred - y_test)')


plt.show()

In [ ]:
city = "NYC"

columns = [
  "residual",
  "prediction"
]

df_random_split = results[2]
df_city = df_random_split[df_random_split["city"] == city]

map = df_city.explore(
    column="residual",
    marker_kwds=dict(radius=5, fill=True),
    cmap="bwr",
    tooltip=False,
    legend=True,
    popup=["log_price", "prediction", "residual"],
    name=f"{city} random split",
    tiles="CartoDB positron"
)
display(map) # show map

In [ ]:
map2= results_spatial[city].explore(
    column="residual",
    marker_kwds=dict(radius=5, fill=True),
    cmap="bwr",
    tooltip=False,
    popup=["log_price", "prediction", "residual"],
    name=f"{city} spatial split",
    tiles="CartoDB positron",
)
display(map2)

In [ ]:
map3 = results_spatial2[city].explore(
    column="residual",
    marker_kwds=dict(radius=5, fill=True),
    cmap="bwr",
    tooltip=False,
    popup=["log_price", "prediction", "residual"],
    name=f"{city} spatial split 2",
    tiles="CartoDB positron",
)
display(map3)